## Trabajo práctico - Pipeline con Aerolinea
## Arquitectura bronza, silver, y gold con spark y delta lake
En este notebook se contruye un pipline de datos para una aerolinea utilizando arquitectura por capas:
-**Bronze: ingest de los archivos fuente
-**Silver: limpieza, tipificaicón u deduplicación
-**Gold: generación de KPI de negocio.


##1. Importamos librerias

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

##2.Creación de catalogos y volumen

In [0]:
catalog = "airline_mantenimiento"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
VOLUMEN_NAME = "landing"

VUELOS_PATH = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/vuelos_diarios.csv"

AERONAVES_PATH = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/aeronaves.csv"

AEROPUERTOS_PATH = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/aeropuertos.csv"
MANTENIMIENTO_PATH = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/mantenimiento.csv"

spark = spark.builder.getOrCreate()

#Creación del catalgo

In [0]:
#creamos el catalogo
spark.sql(f"create catalog if not exists {catalog}")
#Creamos el esquema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{GOLD_SCHEMA}")
#Creamos el volumen
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{BRONZE_SCHEMA}.{VOLUMEN_NAME}")

## 3. Capa Bronze
En esta capa se leen los archivos csv originales se normalizan los tipos básicos y se almacenan como tablas Delta

In [0]:
#lee los vuelos
vuelos_raw = (spark.read
.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/airline_mantenimiento/bronze/landing/vuelos_diarios.csv"))

#lee las aeronaves
aeronaves_raw = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/airline_mantenimiento/bronze/landing/aeronaves.csv"))

#lee las mantenimientos
mantenimiento_raw = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/airline_mantenimiento/bronze/landing/mantenimientos_rds.csv"))

aeropuertos_raw = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/airline_mantenimiento/bronze/landing/aeropuertos.csv"))

In [0]:
#validación
print("Filas vuelos:", vuelos_raw.count())
print("Filas aeronaves:", aeronaves_raw.count())
print("Filas mantenimientos:", mantenimiento_raw.count())
print("Filas Aeropuertos:", aeropuertos_raw.count())

In [0]:
## normalizar vuelos bronze 
vuelos_bronze = (
    vuelos_raw
    .withColumn( "fecha", F.to_date(F.col("fecha")))
    .withColumn("vuelo_id", F.col("vuelo_id") .cast("string"))
    .withColumn("origen_id", F.col("origen_id") .cast("string"))
    .withColumn("destino_id", F.col("destino_id") .cast("string"))
    .withColumn("aeronave_id", F.col("aeronave_id") .cast("string"))
    .withColumn("estado", F.col("estado") .cast("string"))
    .withColumn("duracion_min", F.col("duracion_min") .cast("int"))
    .withColumn("ongestion_time", F.current_timestamp())
    )

In [0]:
## normalización aeronaves
aeronaves_bronze = (
    aeronaves_raw
    .whithColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .whithColumn("modelo_id", F.col("modelo") .Cast("string"))
    .whithColumn("fabriacnte_id", F.col("fabricante").cast("string"))
    .whithColumn("anio_fabricacion_id", F.col("anio_fabricante").cast("int")))

In [0]:
#normalización aeropuertos
aeropuertos_bronze = ((
 aeropuertos_raw
.whithColumn( "aeropuerto_id" , F.Col ("aeropuerto_id") .cast("string"))
.whithColumn("nombre"), F.Col ("nombre") .cast("string"))
.whithColumn("ciudad"), F.Col ("ciudad") .cast("string"))
.whithColumn("pais"), F.Col("pais") .cast("string")
.whithColumn("lat"), F.Col("lat").cast ("double")
.whithColumn("lon"), F.Col("lon").cast("double")

In [0]:
#normalización mantenimiento 
mantenimiento_bronze =((
    mantenimiento_raw
    .whithColumn("mantenimiento_id", F.Col( "mantenimiento_id") .cast("string"))
    .whihColumn("aeronave_id", F.Col("aeronave_id") .cast("string"))
    .whitthColumn("fecha"), F.Col("fecha") .cast("date"))
    .whithtColumn("tipo"), F.Col("tipo") .cast("string"))
    .whithColumn("costo_usd"), F.Col("costo_usd") .cast("double"))
    .whithColumn("duracion_hr"), F.Col("duracion_hr") .cast("double"))

In [0]:
# Guardar tablas bronze
vuelos_bronze.write.format("delta")\
    .mode("ovewrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.vuelos")

    aeronaves_bronze.write.format("delta")\
    .mode("ovewrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.aeronaves")

    mantenimiento_bronze.write.format("delta")\
    .mode("ovewrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.mantenimiento")

    aeropuertos_bronze.write.format("delta")\
    .mode("ovewrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.aeropuertos")

In [0]:
#validación de la tabla
spark.sql(f"select * from {catalog].{BRONZE_SCHEMA}.vuelos").display ()

In [0]:
#### 4.Capa Silver

En esta capa se limpian los datos, se tipifican de nuevo las columnas crítias y se eliminan duplicados 

In [0]:
# leemos las tablas bronze

vuelos_bz = spark.table(f"{catalod}.{bronze_shema}.vuelos")
aeronaves_bz = spark.table(f"{catalog}.{bronze_shema})

In [0]:
# Transformaciones Silver de Vuelos
vuelos_slv = (
    .select(
        "vuelos_id",
        "fecha",
        "origen_id",
        "destino_id",
        "aeronaves_id", 
        "estado", 
        "duracion_min")
)
.withColumn("fechas" F.to_date(F.col("fecha")
.withColumn("vuelos_id", F.col("vuelos_id").cast("string"))
.withColumn("origen_id", F.col("origen_id").cast("string"))
.withColumn("destino_id", F.col("destino_id").cast("string"))
.withColumn("aeronaves_id", F.col("aeronaves_id").cast())



In [0]:
#validacion de calidad
print("Nulos vuelos_id:", vuelos_slv.filter(F.col("vuelo_id").isNull()).count())
print("Duplicados vuelos:", vuelos_slv.groupBy("vuelo_id").count().filter(F.col("count")> 1).conunt())

In [0]:
##5.Capa Gold - KPI de puntualidad de vuelos
Se genera una tabla analítica con métrica de puntualidad por modelo fabircante, pais de origen y periodo.

In [0]:
##Leer silver para la puntalidad
df_vuelos = spark.table(f"{catalog}.{SILVER_SCHEMA}.vuelos")
df_aeronaves = spark.table(f"{catalog}.{SILVER_SCHEMA}.aeronaves")
df_aeropuertos = spark.table(f"{catalog}.{SILVER_SCHEMA}.aeropuertos")



In [0]:
#prepara aeropuerto de origen

df_aeropuertos = df aeropertos.select(
    F.col(("aeropuerto_id").alias("origen_id"),
    F.col("nombre").alias("origen_nombre"),
    F.col("ciudad").alias("origen_ciduad"),
    F.col("pais").alias("origen_pais"))

In [0]:
#Enriquecer los vuelos
df_vuelos_enriquecidos = (
    df_vuelos
    .joint(df_aeropuertos_origen, on = "origen_id", how = "left")
    .joint(df_aeronaves, on = "aeronaves_id", how = "left")
    .withColumn(("anio", F.year("fecha"))
    .withColumn(("mes"), F.month("fecha"))
)
    
#validación
display(df vuelos_enriquecidos)


In [0]:
F.count("vuelo_id").alias("total_vuelos"),
F.count()